# 1.2 层级一：事件流标签解析与宏观阶段划分 - 测试版（决赛）

## 目标

本notebook实现MORPH框架Step 1的层级一：基于事件流标签的宏观阶段划分。

### 核心思想

利用Gradient Sports数据集中已有的精确比赛状态标签（如`setpieceType`、`homeTeam`等），实现**100%准确性**的宏观阶段分类，避免不必要的建模误差。

### 阶段分类

将每一帧分类到以下宏观阶段：
1. **进攻-运动战** (In-Possession Open Play)
2. **防守-运动战** (Out-of-Possession Open Play)
3. **进攻-定位球** (In-Possession Set Pieces)
4. **防守-定位球** (Out-of-Possession Set Pieces)
5. **其他比赛状态** (Other States)

## 数据信息

- **比赛**: 2022世界杯决赛
- **对阵**: 阿根廷 vs 法国
- **gameID**: 10517
- **输入**: 
  - 追踪数据: `data/morph_test/tracking_data_10517.parquet`
  - 事件数据: `Event Data/10517.json`
- **输出**: 
  - 带阶段标签的追踪数据: `data/morph_test/tracking_data_10517_phase.parquet`

## 1. 导入必要的库

In [1]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 数据处理
import polars as pl
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

# 可视化
import matplotlib.pyplot as plt
import seaborn as sns
from mplsoccer import VerticalPitch

# 配置matplotlib中文显示
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'SimSun', 'KaiTi', 'FangSong']
plt.rcParams['axes.unicode_minus'] = False

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pl.Config.set_tbl_rows(100)

print(f"Polars版本: {pl.__version__}")
print("✅ 库导入成功")

Polars版本: 1.2.1
✅ 库导入成功


## 2. 设置数据路径

In [2]:
# 数据根目录
DATA_ROOT = Path(r"E:\JerryWu\Master\SoccerAnalytics\OpenData\TrackingData\Gradient Sports  Enhanced 2022 World Cup Dataset")

# 事件数据目录
EVENT_DIR = DATA_ROOT / "Event Data"

# 决赛数据文件
FINAL_GAME_ID = "10517"
FINAL_EVENT_FILE = EVENT_DIR / f"{FINAL_GAME_ID}.json"

# 输入输出目录
INPUT_DIR = Path("../../data/morph_test")
OUTPUT_DIR = Path("../../data/morph_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 输入文件（来自1.1）
TRACKING_FILE = INPUT_DIR / f"tracking_data_{FINAL_GAME_ID}.parquet"
METADATA_FILE = INPUT_DIR / f"metadata_{FINAL_GAME_ID}.json"

# 验证文件存在
print("检查数据文件...")
files_ok = True

if FINAL_EVENT_FILE.exists():
    print(f"✅ 事件数据: {FINAL_EVENT_FILE.name} ({FINAL_EVENT_FILE.stat().st_size / 1024 / 1024:.2f} MB)")
else:
    print(f"❌ 未找到事件数据: {FINAL_EVENT_FILE}")
    files_ok = False

if TRACKING_FILE.exists():
    print(f"✅ 追踪数据: {TRACKING_FILE.name} ({TRACKING_FILE.stat().st_size / 1024 / 1024:.2f} MB)")
else:
    print(f"❌ 未找到追踪数据: {TRACKING_FILE}")
    files_ok = False

if not files_ok:
    print("\n⚠️  请先运行 1.1_test_Convert_TrackingData.ipynb")

检查数据文件...
✅ 事件数据: 10517.json (21.76 MB)
✅ 追踪数据: tracking_data_10517.parquet (117.18 MB)


## 3. 加载追踪数据

In [3]:
print("加载追踪数据...")
tracking_data = pl.read_parquet(TRACKING_FILE)

print(f"✅ 追踪数据加载成功")
print(f"数据形状: {tracking_data.shape}")
print(f"\n前5行:")
display(tracking_data.head())

加载追踪数据...
✅ 追踪数据加载成功
数据形状: (2966182, 21)

前5行:


period_id,timestamp,frame_id,ball_state,id,x,y,z,team_id,position_name,game_id,vx,vy,vz,v,ax,ay,az,a,ball_owning_team_id,is_ball_carrier
i64,duration[μs],i64,str,str,f64,f64,f64,str,str,i32,f64,f64,f64,f64,f64,f64,f64,f64,str,bool
1,821µs,4630,"""alive""","""10715""",4.987,-1.993,0.0,"""364""","""ST""",10517,-0.522823,0.079386,0.0,0.528815,0.0,0.0,0.0,0.0,"""363""",false
1,821µs,4630,"""alive""","""11""",-18.172,3.632,0.0,"""364""","""LCB""",10517,-1.114241,-0.985899,0.0,1.487794,0.0,0.0,0.0,0.0,"""363""",false
1,821µs,4630,"""alive""","""113""",0.763,-18.099,0.0,"""363""","""ST""",10517,-2.287401,-0.437484,0.0,2.328862,0.0,0.0,0.0,0.0,"""363""",false
1,821µs,4630,"""alive""","""11856""",-6.252,3.173,0.0,"""364""","""CM""",10517,-1.145117,-1.007385,0.0,1.525161,0.0,0.0,0.0,0.0,"""363""",false
1,821µs,4630,"""alive""","""13222""",3.636,18.712,0.0,"""364""","""RB""",10517,-0.874427,-1.040521,0.0,1.359156,0.0,0.0,0.0,0.0,"""363""",false


## 4. 加载事件数据

In [4]:
print("加载事件数据...")
with open(FINAL_EVENT_FILE, 'r', encoding='utf-8') as f:
    event_data_raw = json.load(f)

print(f"✅ 事件数据加载成功")
print(f"事件总数: {len(event_data_raw)}")
print(f"\n第一个事件示例:")
print(json.dumps(event_data_raw[0], indent=2, ensure_ascii=False)[:1000] + "...")

加载事件数据...
✅ 事件数据加载成功
事件总数: 2814

第一个事件示例:
{
  "gameId": 10517,
  "gameEventId": 6737736,
  "possessionEventId": 6622158,
  "startTime": 154.487,
  "endTime": 154.487,
  "duration": 0.0,
  "eventTime": 154.487,
  "sequence": 1.0,
  "gameEvents": {
    "gameEventType": "FIRSTKICKOFF",
    "initialNonEvent": false,
    "startGameClock": 0,
    "startFormattedGameClock": "00:00",
    "period": 1,
    "videoMissing": false,
    "teamId": 363,
    "teamName": "France",
    "homeTeam": false,
    "playerId": 1534,
    "playerName": "Antoine Griezmann",
    "touches": 1,
    "touchesInBox": 0,
    "setpieceType": "K",
    "earlyDistribution": false,
    "videoUrl": "https://epitome.pff.com/en/film_room/0d77770d-6029-4fb0-b3b0-0ea38ac14337/154.487",
    "endType": null,
    "outType": null,
    "subType": null,
    "playerOffId": null,
    "playerOffName": null,
    "playerOffType": null,
    "playerOnId": null,
    "playerOnName": null
  },
  "initialTouch": {
    "initialBodyType": "L",
    "

## 5. 解析事件数据

提取关键字段用于阶段分类：
- `startTime`, `endTime`: 事件时间范围
- `gameEvents.setpieceType`: 定位球类型（O=运动战）
- `gameEvents.homeTeam`: 是否主队控球
- `gameEvents.period`: 比赛时期

In [5]:
print("解析事件数据...")

# 提取关键字段
events_list = []
for event in event_data_raw:
    try:
        game_events = event.get('gameEvents', {})
        
        event_info = {
            'gameEventId': event.get('gameEventId'),
            'startTime': event.get('startTime'),
            'endTime': event.get('endTime'),
            'period': game_events.get('period'),
            'setpieceType': game_events.get('setpieceType'),
            'homeTeam': game_events.get('homeTeam'),
            'gameEventType': game_events.get('gameEventType'),
        }
        events_list.append(event_info)
    except Exception as e:
        print(f"警告: 解析事件失败 - {e}")
        continue

# 转换为DataFrame
events_df = pd.DataFrame(events_list)

print(f"✅ 事件解析完成")
print(f"有效事件数: {len(events_df)}")
print(f"\n事件数据前5行:")
display(events_df.head())

print(f"\n定位球类型分布:")
print(events_df['setpieceType'].value_counts())

解析事件数据...
✅ 事件解析完成
有效事件数: 2814

事件数据前5行:


,gameEventId,startTime,endTime,period,setpieceType,homeTeam,gameEventType
0,6737736,154.487,154.487,1,K,False,FIRSTKICKOFF
1,6737739,155.389,156.690,1,O,False,OTB
2,6737739,155.389,156.690,1,O,False,OTB
3,6737740,161.028,NaN,1,None,None,OUT
4,6737744,170.237,170.237,1,T,True,OTB



定位球类型分布:
setpieceType
O    2491
F      51
T      48
G      19
P      11
C      11
K      10
D       1
Name: count, dtype: int64


## 6. 定义阶段分类函数

根据事件标签将每个事件分类到宏观阶段：

### 分类规则

1. **进攻-运动战**: `setpieceType='O'` AND `homeTeam=True` (对主队)
2. **防守-运动战**: `setpieceType='O'` AND `homeTeam=False` (对主队)
3. **进攻-定位球**: `setpieceType!='O'` AND `homeTeam=True`
4. **防守-定位球**: `setpieceType!='O'` AND `homeTeam=False`
5. **其他状态**: 球出界、比赛暂停等

In [6]:
def classify_phase(row, team='home'):
    """
    根据事件标签分类比赛阶段
    
    Parameters:
    -----------
    row : pd.Series
        事件数据行
    team : str
        'home' 或 'away'，指定从哪个球队的视角分类
    
    Returns:
    --------
    str : 阶段标签
    """
    setpiece = row['setpieceType']
    home_team = row['homeTeam']
    event_type = row['gameEventType']
    
    # 处理缺失值
    if pd.isna(setpiece) or pd.isna(home_team):
        return 'OTHER'
    
    # 判断是否为该队控球
    in_possession = (home_team == True) if team == 'home' else (home_team == False)
    
    # 运动战 (Open Play)
    if setpiece == 'O':
        if in_possession:
            return 'IN_POSS_OPEN_PLAY'
        else:
            return 'OUT_POSS_OPEN_PLAY'
    
    # 定位球 (Set Pieces)
    elif setpiece in ['C', 'F', 'G', 'K', 'P', 'T']:
        if in_possession:
            return 'IN_POSS_SET_PIECE'
        else:
            return 'OUT_POSS_SET_PIECE'
    
    # 其他状态
    else:
        return 'OTHER'

# 应用分类函数（从主队视角）
events_df['phase_home'] = events_df.apply(lambda row: classify_phase(row, team='home'), axis=1)
events_df['phase_away'] = events_df.apply(lambda row: classify_phase(row, team='away'), axis=1)

print("✅ 阶段分类完成")
print(f"\n主队阶段分布:")
print(events_df['phase_home'].value_counts())
print(f"\n客队阶段分布:")
print(events_df['phase_away'].value_counts())

✅ 阶段分类完成

主队阶段分布:
phase_home
IN_POSS_OPEN_PLAY     1340
OUT_POSS_OPEN_PLAY    1136
OTHER                  188
OUT_POSS_SET_PIECE      80
IN_POSS_SET_PIECE       70
Name: count, dtype: int64

客队阶段分布:
phase_away
OUT_POSS_OPEN_PLAY    1340
IN_POSS_OPEN_PLAY     1136
OTHER                  188
IN_POSS_SET_PIECE       80
OUT_POSS_SET_PIECE      70
Name: count, dtype: int64


## 7. 保存带阶段标签的数据

In [7]:
# 注意：由于事件数据和追踪数据的时间对齐较复杂，
# 这里先保存事件级别的阶段标签，供后续步骤使用

# 保存事件阶段数据
output_event_phase = OUTPUT_DIR / f"event_phases_{FINAL_GAME_ID}.csv"
events_df.to_csv(output_event_phase, index=False, encoding='utf-8')
print(f"✅ 事件阶段数据已保存: {output_event_phase}")

print("\n" + "=" * 60)
print("层级一：宏观阶段划分完成！")
print("=" * 60)
print(f"\n输出文件:")
print(f"  - 事件阶段标签: {output_event_phase}")
print(f"\n✅ 数据已准备就绪，可以进行层级二的细粒度意图识别")

✅ 事件阶段数据已保存: ..\..\data\morph_test\event_phases_10517.csv

层级一：宏观阶段划分完成！

输出文件:
  - 事件阶段标签: ..\..\data\morph_test\event_phases_10517.csv

✅ 数据已准备就绪，可以进行层级二的细粒度意图识别


## 8. 总结

本notebook完成了以下工作：

1. ✅ 加载事件数据和追踪数据
2. ✅ 解析事件流标签（setpieceType, homeTeam等）
3. ✅ 实现宏观阶段分类函数
4. ✅ 对所有事件进行阶段标注
5. ✅ 保存阶段标签数据

### 阶段分类结果

- **进攻-运动战**: 球队控球且为运动战
- **防守-运动战**: 对手控球且为运动战
- **进攻-定位球**: 球队控球且为定位球
- **防守-定位球**: 对手控球且为定位球
- **其他状态**: 球出界、比赛暂停等

### 下一步

- 运行 `1.3_test_Tactical_Intent.ipynb` 进行层级二的细粒度战术意图识别
- 运行 `1.4_test_Scaling.ipynb` 进行空间对齐